In [ ]:
import math
import random
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors

from dataset import preprocess_pipeline
from model import Encoder, Decoder, DomainClassifier, GMM_VAE
from train import pretrain_vae, initialize_gmm_parameters, train_joint_gmm_vae

warnings.filterwarnings("ignore")
sns.set_theme(style="white", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "axes.spines.top": False,
                     "axes.spines.right": False})

CFG = dict(
    rna_path="/Users/doanlong/Documents/GMM_VAE/data/adata_rna.h5ad",
    spa_path="/Users/doanlong/Documents/GMM_VAE/data/adata_spa.h5ad",
    label_col="cell_type",
    n_top_genes=2000,
    batch_size=256,
    latent_dim=32,
    lr_pretrain=1e-3,
    lr_joint=5e-4,
    epochs_pretrain=30,
    epochs_joint=50,
    lambda_1=0.1,
    lambda_2=1.0,
    max_cells_plot=8000,
    seed=42,
)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Read Data

In [ ]:
adata_rna = sc.read_h5ad(CFG["rna_path"])
adata_spa = sc.read_h5ad(CFG["spa_path"])

dl_rna, dl_spa, label_mapping = preprocess_pipeline(
    adata_rna, adata_spa,
    label_col=CFG["label_col"],
    n_top_genes=CFG["n_top_genes"],
    batch_size=CFG["batch_size"],
)

ds_rna, ds_spa = dl_rna.dataset, dl_spa.dataset
idx_to_label = {v: k for k, v in label_mapping.items()}
class_names = [idx_to_label[i] for i in range(len(idx_to_label))]

input_dim = ds_rna.data.shape[1]
n_clusters = len(label_mapping)
print(f"scRNA-seq: {ds_rna.data.shape[0]} cells | ST: {ds_spa.data.shape[0]} cells | "
      f"genes: {input_dim} | cell types (source): {n_clusters}")

/Users/doanlong/Documents/GMM_VAE/Methods/preprocess.py:28: UserWarning: `n_top_genes` (=351) > number of normalized dispersions (=349), returning all genes with normalized dispersions.
  sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, subset=True)


# Khởi tạo mô hình

In [ ]:
def build_hidden_dims(input_dim, latent_dim, n_layers=3, min_dim=64):
    dims, h = [], input_dim
    for _ in range(n_layers):
        h = max(h // 2, min_dim)
        if h <= latent_dim:
            break
        dims.append(h)
    return tuple(dims) if dims else (max(min_dim, 2 * latent_dim),)

latent_dim = CFG["latent_dim"]
hidden_dims = build_hidden_dims(input_dim, latent_dim)

encoder = Encoder(input_dim, hidden_dims, latent_dim)
decoder_src = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)
decoder_tgt = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)
domain_clf = DomainClassifier(latent_dim)

model = GMM_VAE(encoder, decoder_src, decoder_tgt, domain_clf,
                n_clusters=n_clusters, latent_dim=latent_dim).to(device)

optimizer_pre = torch.optim.Adam(model.parameters(), lr=CFG["lr_pretrain"])

print(f"hidden_dims: {hidden_dims}")
print(f"Tổng tham số: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
def build_hidden_dims(input_dim, latent_dim, n_layers=3, min_dim=64):
    dims, h = [], input_dim
    for _ in range(n_layers):
        h = max(h // 2, min_dim)
        if h <= latent_dim:
            break
        dims.append(h)
    return tuple(dims) if dims else (max(min_dim, 2 * latent_dim),)

latent_dim = CFG["latent_dim"]
hidden_dims = build_hidden_dims(input_dim, latent_dim)

encoder = Encoder(input_dim, hidden_dims, latent_dim)
decoder_src = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)
decoder_tgt = Decoder.from_encoder_dims(hidden_dims, latent_dim, input_dim)
domain_clf = DomainClassifier(latent_dim)

model = GMM_VAE(encoder, decoder_src, decoder_tgt, domain_clf,
                n_clusters=n_clusters, latent_dim=latent_dim).to(device)

optimizer_pre = torch.optim.Adam(model.parameters(), lr=CFG["lr_pretrain"])

print(f"hidden_dims: {hidden_dims}")
print(f"Tổng tham số: {sum(p.numel() for p in model.parameters()):,}")

# Train model

In [ ]:
hist_pre = pretrain_vae(model, dl_rna, optimizer_pre, CFG["epochs_pretrain"], device)

plt.figure(figsize=(5, 3.2))
plt.plot(range(1, len(hist_pre) + 1), hist_pre, color="#3b6fb6", lw=2)
plt.xlabel("Epoch"); plt.ylabel("Reconstruction loss"); plt.title("Pre-training")
plt.tight_layout(); plt.show()

## K-Means trên mu của scRNA-seq để khởi tạo pi, mu_c, var_c

In [ ]:
info = initialize_gmm_parameters(model, dl_rna, n_clusters, device, seed=CFG["seed"])
label_to_cluster = info["label_to_cluster"]

assert (label_to_cluster >= 0).all(), "Có nhãn chưa được ghép cụm: kiểm tra n_clusters"
print("Ánh xạ nhãn -> cụm GMM:")
for lbl, clu in enumerate(label_to_cluster.tolist()):
    print(f"  {class_names[lbl]:<25} -> cụm {clu}")
print("pi:", np.round(info["pi"].cpu().numpy(), 3))

## huấn luyện chung. Tạo optimizer mới

In [ ]:
optimizer_joint = torch.optim.Adam(model.parameters(), lr=CFG["lr_joint"])

def grl_lambda(progress, gamma=10.0):
    return 2.0 / (1.0 + math.exp(-gamma * progress)) - 1.0

E = CFG["epochs_joint"]
hist_joint = []
for ep in range(E):
    lam = grl_lambda(ep / max(E - 1, 1))
    h = train_joint_gmm_vae(
        model, dl_rna, dl_spa, optimizer_joint,
        epochs=1,
        lambda_1=CFG["lambda_1"], lambda_2=CFG["lambda_2"], lambda_grl=lam,
        device=device,
        label_to_cluster=label_to_cluster,
    )[0]
    h["lambda_grl"] = lam
    hist_joint.append(h)

hist_df = pd.DataFrame(hist_joint)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.2))
for ax, (col, color) in zip(axes, [("recon", "#3b6fb6"), ("gmm", "#d9822b"),
                                    ("domain", "#c0392b"), ("lambda_grl", "#2e8b57")]):
    ax.plot(hist_df.index + 1, hist_df[col], color=color, lw=2)
    ax.set_title(col); ax.set_xlabel("Epoch")
axes[2].axhline(math.log(2), ls="--", c="gray", lw=1)
axes[2].text(0.98, 0.9, "ln 2 (đoán ngẫu nhiên)", transform=axes[2].transAxes,
             ha="right", fontsize=8, color="gray")
plt.tight_layout(); plt.show()

## Trích xuất không gian ẩn

In [ ]:
@torch.no_grad()
def extract_latent(model, dataset, batch_size, device):
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=False)
    zs, ys, gs = [], [], []
    for x, y, _ in loader:
        mu, _ = model.encoder(x.to(device))
        zs.append(mu.cpu())
        gs.append(model.get_gamma(mu).cpu())
        ys.append(y)
    return torch.cat(zs).numpy(), torch.cat(ys).numpy(), torch.cat(gs).numpy()

model.eval()
Z_rna, y_rna, gamma_rna = extract_latent(model, ds_rna, CFG["batch_size"], device)
Z_spa, y_spa, gamma_spa = extract_latent(model, ds_spa, CFG["batch_size"], device)
print("Z_rna:", Z_rna.shape, "| Z_spa:", Z_spa.shape)

# Evaluate

In [ ]:
rng = np.random.default_rng(CFG["seed"])
n_plot = min(CFG["max_cells_plot"], len(Z_rna), len(Z_spa))
idx_r = np.sort(rng.choice(len(Z_rna), n_plot, replace=False))
idx_s = np.sort(rng.choice(len(Z_spa), n_plot, replace=False))

Z_plot = np.vstack([Z_rna[idx_r], Z_spa[idx_s]])
domain = np.array(["scRNA-seq"] * n_plot + ["Spatial (ST)"] * n_plot)
is_rna = domain == "scRNA-seq"

ad = sc.AnnData(Z_plot.astype(np.float32))
sc.pp.neighbors(ad, n_neighbors=15, use_rep="X")
sc.tl.umap(ad, random_state=CFG["seed"])
emb_umap = ad.obsm["X_umap"]

emb_tsne = TSNE(n_components=2, perplexity=30, init="pca",
                random_state=CFG["seed"], n_jobs=-1).fit_transform(Z_plot)

embeddings = {"UMAP": emb_umap, "t-SNE": emb_tsne}

In [ ]:
def style_axis(ax, title, name):
    ax.set_title(title, fontsize=12, weight="bold")
    ax.set_xlabel(f"{name} 1"); ax.set_ylabel(f"{name} 2")
    ax.set_xticks([]); ax.set_yticks([])

def scatter_cat(ax, emb, labels, palette, s=5, alpha=0.8):
    for lab, color in palette.items():
        m = labels == lab
        ax.scatter(emb[m, 0], emb[m, 1], s=s, c=[color], alpha=alpha,
                   linewidths=0, rasterized=True, label=lab)

domain_palette = {"scRNA-seq": "#3b6fb6", "Spatial (ST)": "#e07b39"}

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
order = rng.permutation(len(domain))
for ax, (name, emb) in zip(axes, embeddings.items()):
    for dom, color in domain_palette.items():
        pass
    ax.scatter(emb[order, 0], emb[order, 1], s=5, linewidths=0, rasterized=True, alpha=0.7,
               c=[domain_palette[d] for d in domain[order]])
    style_axis(ax, f"Domain alignment ({name})", name)

handles = [plt.Line2D([], [], marker="o", ls="", color=c, label=k, markersize=7)
           for k, c in domain_palette.items()]
fig.legend(handles=handles, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.02))
plt.tight_layout(); plt.savefig("fig1_domain_alignment.png", bbox_inches="tight"); plt.show()

In [ ]:
labels_rna_plot = np.array([class_names[i] for i in y_rna[idx_r]])
type_palette = dict(zip(class_names, sns.color_palette("husl", len(class_names))
                        if len(class_names) > 20 else sns.color_palette("tab20", len(class_names))))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for ax, (name, emb) in zip(axes, embeddings.items()):
    scatter_cat(ax, emb[is_rna], labels_rna_plot, type_palette)
    style_axis(ax, f"scRNA-seq theo Ground Truth ({name})", name)
axes[1].legend(loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False,
               markerscale=3, fontsize=9, title="Cell type")
plt.tight_layout(); plt.savefig("fig2_source_celltype.png", bbox_inches="tight"); plt.show()

In [ ]:
g = gamma_spa[idx_s]
comp = g.argmax(1)
entropy = -(g * np.log(g + 1e-10)).sum(1)
cluster_to_label = {int(c): class_names[l] for l, c in enumerate(label_to_cluster.tolist())}

emb = embeddings["UMAP"][~is_rna]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))

comp_palette = dict(zip(range(n_clusters), sns.color_palette("tab20", n_clusters)))
for c, color in comp_palette.items():
    m = comp == c
    axes[0].scatter(emb[m, 0], emb[m, 1], s=5, c=[color], linewidths=0, alpha=0.8,
                    rasterized=True, label=f"{c}: {cluster_to_label.get(c, '?')}")
style_axis(axes[0], "ST theo thành phần GMM (argmax γ)", "UMAP")
axes[0].legend(loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False,
               markerscale=3, fontsize=8, title="Cụm: loại nguồn tương ứng")

sc_ = axes[1].scatter(emb[:, 0], emb[:, 1], s=5, c=entropy, cmap="magma",
                      linewidths=0, alpha=0.85, rasterized=True)
style_axis(axes[1], "ST theo Entropy của γ", "UMAP")
cb = plt.colorbar(sc_, ax=axes[1], fraction=0.046, pad=0.02); cb.set_label("Entropy")

plt.tight_layout(); plt.savefig("fig3_target_structure.png", bbox_inches="tight"); plt.show()

In [ ]:
km_pred = KMeans(n_clusters=n_clusters, n_init=10, random_state=CFG["seed"]).fit_predict(Z_rna)
gmm_pred = gamma_rna.argmax(1)

rows = []
for name, pred in [("K-Means trên Z", km_pred), ("GMM (argmax γ)", gmm_pred)]:
    rows.append({"Phương pháp": name,
                 "ARI": adjusted_rand_score(y_rna, pred),
                 "NMI": normalized_mutual_info_score(y_rna, pred),
                 "Số cụm dùng": len(np.unique(pred))})

metrics_df = pd.DataFrame(rows).set_index("Phương pháp").round(4)
display(metrics_df)

ax = metrics_df[["ARI", "NMI"]].plot.bar(figsize=(5.5, 3.5), color=["#3b6fb6", "#e07b39"], rot=0, width=0.7)
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.set_title("Chất lượng phân cụm không gian ẩn (scRNA-seq)")
for c in ax.containers:
    ax.bar_label(c, fmt="%.3f", fontsize=9, padding=2)
plt.tight_layout(); plt.show()

In [ ]:
nn_ = NearestNeighbors(n_neighbors=31).fit(Z_plot)
_, nbr = nn_.kneighbors(Z_plot)
nbr = nbr[:, 1:]
other_frac = (is_rna[nbr] != is_rna[:, None]).mean(1)

print(f"Tỉ lệ láng giềng khác miền (trung bình): {other_frac.mean():.3f}  (lý tưởng ≈ 0.5)")
print(f"  - scRNA-seq: {other_frac[is_rna].mean():.3f} | ST: {other_frac[~is_rna].mean():.3f}")

In [ ]:
torch.save({"model": model.state_dict(), "label_mapping": label_mapping,
            "config": CFG, "hidden_dims": hidden_dims}, "gmm_vae_stage12.pt")
np.savez("latent_stage12.npz", Z_rna=Z_rna, y_rna=y_rna, gamma_rna=gamma_rna,
         Z_spa=Z_spa, gamma_spa=gamma_spa)